# Monitor de Riesgo Reputacional

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Monitorizar menciones** en medios y redes con `CORTEX.SENTIMENT()`
2. **Detectar crisis reputacionales** mediante alertas de sentimiento
3. **Clasificar fuentes** por impacto y alcance
4. **Generar reportes ejecutivos** con `CORTEX.COMPLETE()`
5. **Construir dashboard** de riesgo reputacional en tiempo real

---

## Lo Que Construirás

Un **sistema de monitorización reputacional**:
- Análisis de sentimiento de menciones en medios
- Detección de picos negativos (alertas de crisis)
- Clasificación por fuente, tema e impacto
- Reportes ejecutivos automáticos

**Valor de Negocio**: Detectar y responder a crisis reputacionales antes de que escalen.

---

## Funcionalidades Clave

- `CORTEX.SENTIMENT()` — Scoring de sentimiento
- `CORTEX.COMPLETE()` — Clasificación y reporting
- Window functions — Detección de tendencias y picos

¡Comencemos!

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS RIESGO_REPUTACIONAL_DB;
CREATE SCHEMA IF NOT EXISTS RIESGO_REPUTACIONAL_DB.PUBLIC;
USE SCHEMA RIESGO_REPUTACIONAL_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Menciones Sintéticas

### Qué Vamos a Crear

- **2.000 menciones** de medios y redes sociales
- **5 temas**: Servicio, Productos, Sostenibilidad, Escándalos, Tecnología
- **Mezcla** de sentimiento positivo, negativo y neutro

In [ ]:
CREATE OR REPLACE TABLE MENCIONES (
    mencion_id VARCHAR(10) PRIMARY KEY,
    fuente VARCHAR(30),
    fecha TIMESTAMP,
    tema VARCHAR(30),
    texto TEXT,
    alcance INTEGER
);

INSERT INTO MENCIONES
SELECT
    'MEN' || LPAD(SEQ4()::STRING, 5, '0'),
    ARRAY_CONSTRUCT('Twitter','Prensa','LinkedIn','Foro','Blog','TV')[UNIFORM(0,5,RANDOM())]::VARCHAR,
    DATEADD('hour', -UNIFORM(0, 2160, RANDOM()), CURRENT_TIMESTAMP()),
    ARRAY_CONSTRUCT('Servicio','Productos','Sostenibilidad','Escándalo','Tecnología')[UNIFORM(0,4,RANDOM())]::VARCHAR,
    CASE UNIFORM(1,10,RANDOM())
        WHEN 1 THEN 'Increíble lo mal que funciona la app del banco. Llevo tres días sin poder acceder a mi cuenta. Servicio tercermundista.'
        WHEN 2 THEN 'Muy contento con el nuevo fondo de inversión ESG. Rentabilidad del 8% y alineado con valores sostenibles.'
        WHEN 3 THEN 'URGENTE: Filtración de datos de 10.000 clientes del banco. La AEPD ya investiga. Grave brecha de seguridad.'
        WHEN 4 THEN 'El banco ha anunciado el cierre de 50 oficinas rurales. Los pueblos se quedan sin servicios financieros básicos.'
        WHEN 5 THEN 'Excelente experiencia con el asesor financiero. Me explicó todo con paciencia y encontramos la mejor opción hipotecaria.'
        WHEN 6 THEN 'Las comisiones de mantenimiento son abusivas. 8€/mes por una cuenta básica es un robo. Me cambio de banco.'
        WHEN 7 THEN 'Reconocimiento internacional: el banco ha sido premiado como mejor banca digital de Europa por tercer año consecutivo.'
        WHEN 8 THEN 'Escándalo de blanqueo de capitales: directivos implicados en operaciones con paraísos fiscales según investigación periodística.'
        WHEN 9 THEN 'La nueva tarjeta eco de plástico reciclado es una gran iniciativa. Pequeños gestos que marcan la diferencia.'
        ELSE 'El tipo de interés de la hipoteca variable ha subido un 2%. Muchas familias están sufriendo para llegar a fin de mes.'
    END,
    UNIFORM(100, 500000, RANDOM())
FROM TABLE(GENERATOR(ROWCOUNT => 2000));

SELECT tema, COUNT(*) FROM MENCIONES GROUP BY 1;

---

## Paso 3: Análisis de Sentimiento y Clasificación

### Pipeline de Análisis

1. Scoring de sentimiento con `CORTEX.SENTIMENT()`
2. Clasificación de impacto con `CORTEX.COMPLETE()`
3. Detección de picos negativos

In [ ]:
CREATE OR REPLACE TABLE ANALISIS_MENCIONES AS
SELECT
    mencion_id, fuente, fecha, tema, texto, alcance,
    SNOWFLAKE.CORTEX.SENTIMENT(texto) AS score_sentimiento,
    CASE
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(texto) < -0.5 THEN 'Crisis'
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(texto) < -0.2 THEN 'Negativo'
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(texto) < 0.2 THEN 'Neutro'
        ELSE 'Positivo'
    END AS nivel,
    CASE
        WHEN alcance > 100000 AND SNOWFLAKE.CORTEX.SENTIMENT(texto) < -0.3 THEN 'ALTA'
        WHEN alcance > 50000 OR SNOWFLAKE.CORTEX.SENTIMENT(texto) < -0.5 THEN 'MEDIA'
        ELSE 'BAJA'
    END AS alerta
FROM MENCIONES;

SELECT nivel, alerta, COUNT(*), ROUND(AVG(score_sentimiento),3) AS score_medio
FROM ANALISIS_MENCIONES GROUP BY 1, 2 ORDER BY 1, 2;

---

## Paso 4: Reporte Ejecutivo IA

In [ ]:
-- Generar reporte diario
SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
    'Genera un reporte ejecutivo de riesgo reputacional para el comité de dirección con estos datos:\n\n' ||
    'Menciones crisis: ' || (SELECT COUNT(*) FROM ANALISIS_MENCIONES WHERE nivel = 'Crisis') || '\n' ||
    'Menciones negativas: ' || (SELECT COUNT(*) FROM ANALISIS_MENCIONES WHERE nivel = 'Negativo') || '\n' ||
    'Menciones positivas: ' || (SELECT COUNT(*) FROM ANALISIS_MENCIONES WHERE nivel = 'Positivo') || '\n' ||
    'Alertas altas: ' || (SELECT COUNT(*) FROM ANALISIS_MENCIONES WHERE alerta = 'ALTA') || '\n' ||
    'Temas principales: ' || (SELECT LISTAGG(DISTINCT tema, ', ') FROM ANALISIS_MENCIONES WHERE nivel = 'Crisis') || '\n' ||
    'Genera 4 párrafos: resumen, riesgos identificados, acciones recomendadas, perspectiva. En español.'
) AS reporte_ejecutivo;

---

## Paso 5: Dashboard Reputacional

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Monitor de Riesgo Reputacional')
st.markdown('### Monitorización en Tiempo Real')

total = session.sql('SELECT COUNT(*) FROM ANALISIS_MENCIONES').collect()[0][0]
crisis = session.sql("SELECT COUNT(*) FROM ANALISIS_MENCIONES WHERE nivel='Crisis'").collect()[0][0]
score = session.sql('SELECT ROUND(AVG(score_sentimiento),3) FROM ANALISIS_MENCIONES').collect()[0][0]

c1, c2, c3 = st.columns(3)
c1.metric('Total Menciones', f'{total:,}')
c2.metric('Crisis', f'{crisis:,}')
c3.metric('Sentimiento Medio', f'{score}')

st.markdown('---')
st.subheader('Distribución de Sentimiento')
df = session.sql('SELECT nivel, COUNT(*) AS n FROM ANALISIS_MENCIONES GROUP BY 1').to_pandas()
st.bar_chart(df.set_index('NIVEL'))

st.markdown('---')
st.subheader('Alertas Activas')
df_alert = session.sql("SELECT mencion_id, fuente, tema, texto, score_sentimiento, alerta FROM ANALISIS_MENCIONES WHERE alerta IN ('ALTA','MEDIA') ORDER BY score_sentimiento LIMIT 15").to_pandas()
st.dataframe(df_alert)

---

## Paso 6: Limpieza (Opcional)

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS RIESGO_REPUTACIONAL_DB;